# Phase 2 — MPS/MPO Tensor-Network Compression

**Project**: Quantum-Inspired Tensor-Network Compression of OpenVLA-7B  
**Challenge**: Global Quantum + AI Challenge 2026 — VW Group Enterprise Track  
**Phase**: 2 of 7 | **Execution**: Google Colab (GPU runtime required)

---

## What this notebook does

Implements the **quantum-inspired component** required by challenge §5.2:  
MPS (Matrix Product State) decomposition of OpenVLA-7B weight matrices, following
the CompactifAI [[arXiv:2401.14109]](https://arxiv.org/abs/2401.14109) methodology.

**Algorithm** (executed once per target layer per bond dimension):
1. Reshape weight matrix W ∈ R^(m×n) into a 4-site tensor [d₀, d₁, d₂, d₃]
2. Apply sequential SVD sweep (left → right), truncating each bond to χ singular values
3. Store MPS cores; validate with **quimb** (TN bookkeeping)
4. Reconstruct approximate W_hat by contracting the cores
5. Replace layer weight with W_hat and, optionally, re-quantize to INT8

**Bond dimensions swept**: χ ∈ {16, 32, 64, 128}

**Compression targets**: `q_proj`, `k_proj`, `v_proj`, `o_proj`,
`gate_proj`, `up_proj`, `down_proj` in the LLM backbone (~224 layers)

**Checkpoint format**: MPS cores are saved to `checkpoints/compressed_chi{X}/cores.pt`
(compact — 50–200 MB per χ, vs. ~14 GB for a full FP16 model copy).
Phase 3 reconstructs W_hat from cores at evaluation time.

## Before running

1. Set runtime to **GPU** (Runtime → Change runtime type).  
   **Colab Pro / A100 recommended** — FP16 model loading uses ~14 GB VRAM.
   If you only have a T4 (16 GB), it will be very tight; see the fallback note in cell 8.
2. Set `HF_TOKEN` in Colab Secrets (left panel → key icon).
3. Fill in `REPO_URL` in cell 3 with your GitHub repo URL.
4. This notebook can run one or two bond dimensions per Colab session.  
   Set `BOND_DIMS_THIS_RUN` in cell 7 to control which χ values to process.

---

> **License notice** — OpenVLA-7B *model weights* are subject to the
> **LLaMA-2 Community License** (non-commercial research use only).

In [ ]:
# ── Cell 1: Install / pin dependencies ───────────────────────────────────────────
# Uses subprocess so pip runs in the current Python env before any imports.
# OpenVLA version pins (from openvla/openvla pyproject.toml):
#   transformers==4.40.1  — only version tested by OpenVLA authors
#   tokenizers==0.19.1   — must match transformers 4.40.1
#   timm==0.9.10         — imported by modeling_prismatic.py via trust_remote_code
#   sentencepiece==0.1.99 — required by the LLaMA-2 tokenizer
import subprocess, sys

def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

_pip(
    "transformers==4.40.1",
    "tokenizers==0.19.1",
    "timm==0.9.10",
    "sentencepiece==0.1.99",
)
_pip(
    "accelerate>=0.27.0",
    "bitsandbytes>=0.43.0",
    "quimb>=1.8.0",
    "opt-einsum>=3.3.0",
    "pynvml",
    "PyYAML",
    "tqdm",
)
print("pip installs complete.")

In [ ]:
# ── Cell 2: Verify transformers version ────────────────────────────────────────
# Fails loudly before the 15 GB model download if the wrong version landed.
# If this cell raises: Runtime -> Disconnect and delete runtime, then re-run
# from Cell 1 without running anything else first.
import transformers

REQUIRED_TRANSFORMERS = "4.40.1"
_ver = transformers.__version__
print(f"transformers : {_ver}")

if _ver != REQUIRED_TRANSFORMERS:
    raise RuntimeError(
        f"transformers {_ver} installed but OpenVLA requires exactly {REQUIRED_TRANSFORMERS}.\n"
        f"Other 4.x builds and all 5.x builds break modeling_prismatic.py\n"
        f"(_supports_sdpa missing, is_flax_available moved, etc.).\n\n"
        f"Fix:\n"
        f"  1. Runtime -> Disconnect and delete runtime\n"
        f"  2. Re-open the notebook and run Cell 1 before anything else\n"
        f"  3. Do NOT run any other cells before Cell 1 finishes"
    )
print(f"  confirmed: {REQUIRED_TRANSFORMERS}")

In [ ]:
# ── Cell 3: Clone repo and install the vlam_compress package ─────────────────
# The repo is private — authenticates via the GH_TOKEN Colab Secret.
# Add a GitHub PAT (repo scope) as GH_TOKEN in Colab Secrets before running.
import os, sys, subprocess

REPO_DIR = "/content/vlam"

try:
    from google.colab import userdata
    _gh_token = userdata.get("GH_TOKEN")
    REPO_URL = f"https://{_gh_token}@github.com/quantumadventurer11/vw-quantum-vlam-challenge"
    print("GH_TOKEN loaded from Colab Secrets.")
except Exception as _e:
    print(f"GH_TOKEN not found in secrets ({_e}).")
    print("Falling back to unauthenticated URL — will prompt for credentials.")
    REPO_URL = "https://github.com/quantumadventurer11/vw-quantum-vlam-challenge"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print(f"Working directory : {os.getcwd()}")
print("vlam_compress package installed.")

In [ ]:
# ── HuggingFace authentication ────────────────────────────────────────────────
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
    print("Logged in via Colab Secrets.")
except Exception as e:
    print(f"Secret not found ({e}). Falling back to interactive login...")
    login()

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import json
import platform
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import transformers
import yaml
from tqdm.auto import tqdm

import quimb
import quimb.tensor as qtn

from vlam_compress.compress import (
    choose_reshape_dims,
    mps_decompose,
    mps_reconstruct,
    count_core_params,
    frobenius_error,
    find_compression_targets,
    DEFAULT_TARGET_SUFFIXES,
)

print(f"PyTorch    : {torch.__version__}")
print(f"quimb      : {quimb.__version__}")
print(f"Transforms : {transformers.__version__}")
print("All imports OK.")

In [ ]:
# ── Environment check ─────────────────────────────────────────────────────────
assert torch.cuda.is_available(), "No GPU found — change runtime to GPU."

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1024**3

print(f"GPU          : {props.name}")
print(f"VRAM total   : {vram_gb:.1f} GB")
print(f"CUDA         : {torch.version.cuda}")
print(f"Platform     : {platform.platform()}")

if vram_gb < 14:
    print("\n⚠  WARNING: <14 GB VRAM detected. FP16 7B model (~14 GB) may OOM.")
    print("   Options: (a) switch to Colab Pro / A100,")
    print("            (b) set USE_INT8_FOR_COMPRESSION=True in the config cell.")

In [ ]:
# ── Cell 5: Mount Google Drive for checkpoint storage ────────────────────
# Checkpoints (~50-200 MB per chi) persist on Drive across Colab sessions.
# Phase 3 reads from the same Drive path — no manual download/upload needed.
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINTS_BASE = Path("/content/drive/MyDrive/vlam_checkpoints")
    print("Google Drive mounted.")
except Exception as _e:
    print(f"Drive mount skipped ({_e}) — using local checkpoints/")
    CHECKPOINTS_BASE = Path("checkpoints")

CHECKPOINTS_BASE.mkdir(parents=True, exist_ok=True)
print(f"Checkpoints dir  : {CHECKPOINTS_BASE}")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
with open("configs/seeds.yaml") as f:
    cfg = yaml.safe_load(f)

ALL_BOND_DIMS = cfg["bond_dims"]          # [16, 32, 64, 128]

# ▼ Set this to the χ values you want to process in THIS Colab session.
#   If Drive storage is available, you can run all four at once.
#   Otherwise split across sessions: [16, 32] then [64, 128].
BOND_DIMS_THIS_RUN = ALL_BOND_DIMS        # or e.g. [64, 128]

MODEL_ID               = "openvla/openvla-7b"
USE_INT8_FOR_COMPRESSION = False          # True if VRAM < 14 GB (T4 with tight budget)
N_SITES                = 4               # MPS sites per weight matrix
CHECKPOINTS_DIR        = CHECKPOINTS_BASE
RESULTS_DIR            = Path("results")
CHECKPOINTS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Bond dims this run : {BOND_DIMS_THIS_RUN}")
print(f"Model              : {MODEL_ID}")
print(f"MPS sites          : {N_SITES}")
print(f"Load INT8          : {USE_INT8_FOR_COMPRESSION}")

In [ ]:
# ── Load OpenVLA-7B ───────────────────────────────────────────────────────────
# FP16 is used for compression (simplifies in-place weight replacement).
# If VRAM is insufficient, set USE_INT8_FOR_COMPRESSION=True in the config cell.
from transformers import AutoModelForVision2Seq, AutoProcessor

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Loading model (may take 10-20 min on first run)...")
torch.cuda.reset_peak_memory_stats()

if USE_INT8_FOR_COMPRESSION:
    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        attn_implementation="eager",
        load_in_8bit=True,
        device_map="auto",
        trust_remote_code=True,
    )
    print("  Mode: INT8 (memory-efficient but requires dequantization before SVD)")
else:
    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        attn_implementation="eager",
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    print("  Mode: FP16")

model.eval()
peak_mem_mib = torch.cuda.max_memory_allocated() / 1024**2
print(f"Loaded. Peak GPU memory: {peak_mem_mib:.0f} MiB ({peak_mem_mib/1024:.1f} GiB)")

In [ ]:
# ── Identify compression targets ──────────────────────────────────────────────
targets = find_compression_targets(
    model,
    target_suffixes=DEFAULT_TARGET_SUFFIXES,
    min_params=1_000_000,
)

n_target_params = sum(m.weight.numel() for m in targets.values())
n_total_params  = sum(p.numel() for p in model.parameters())

print(f"Target layers        : {len(targets)}")
print(f"Target param count   : {n_target_params/1e9:.3f}B  "
      f"({n_target_params/n_total_params*100:.1f}% of model)")
print(f"Total model params   : {n_total_params/1e9:.3f}B")
print(f"Non-target (fixed)   : {(n_total_params - n_target_params)/1e9:.3f}B")
print()

# Show the unique weight shapes that will be compressed
shapes = {}
for name, mod in targets.items():
    sh = tuple(mod.weight.shape)
    shapes[sh] = shapes.get(sh, 0) + 1

print("Weight shapes and reshape plans:")
for sh, count in sorted(shapes.items()):
    rdims = choose_reshape_dims(sh[0], sh[1], N_SITES)
    print(f"  {sh!s:20s}  ×{count:3d} layers  →  reshape {rdims}")

In [ ]:
# ── quimb TN validation (conceptual demo) ────────────────────────────────────
# This cell demonstrates how quimb represents the MPS structure and validates
# our torch-based decomposition against quimb's contraction engine.
# (quimb is used for TN bookkeeping; torch.linalg.svd handles the computation.)

# Use a small synthetic matrix for the demo (avoids touching model weights).
DEMO_SHAPE = (64, 64)   # tiny square matrix
DEMO_CHI   = 8

W_demo = torch.randn(DEMO_SHAPE, dtype=torch.float32)
rdims  = choose_reshape_dims(*DEMO_SHAPE, n_sites=N_SITES)

print(f"Demo matrix shape : {DEMO_SHAPE}")
print(f"Reshape dims      : {rdims}")
print(f"Bond dim χ        : {DEMO_CHI}")

# Decompose with our torch implementation
cores, sinfo = mps_decompose(W_demo, DEMO_CHI, N_SITES)
W_hat_torch  = mps_reconstruct(cores, DEMO_SHAPE)
err_torch    = frobenius_error(W_demo, W_hat_torch)

print(f"\nTorch reconstruction error : {err_torch:.4%}")
print(f"Core shapes                : {[list(c.shape) for c in cores]}")
print(f"Actual ranks               : {sinfo['actual_ranks']}")

# Build equivalent quimb MPS and contract to verify
np_cores = [c.cpu().float().numpy() for c in cores]

# quimb MatrixProductState convention: arrays are (left_bond, phys, right_bond)
# Our cores already follow this convention.
mps_qu = qtn.MatrixProductState(
    arrays=np_cores,
    shape="lpr",              # left-physical-right
)
W_hat_quimb = mps_qu.to_dense().reshape(rdims).reshape(DEMO_SHAPE)
err_quimb   = float(np.linalg.norm(W_demo.numpy() - W_hat_quimb)
                    / (np.linalg.norm(W_demo.numpy()) + 1e-12))

print(f"\nquimb reconstruction error : {err_quimb:.4%}")
max_diff = float(np.abs(W_hat_torch.numpy() - W_hat_quimb).max())
print(f"Max element diff (torch vs quimb): {max_diff:.2e}")
assert max_diff < 1e-3, "torch and quimb reconstructions disagree — check core convention"
print("[OK] torch and quimb reconstructions match.")
print(f"\nMPS bond structure: {mps_qu}")

In [ ]:
# ── Weight extraction helper ──────────────────────────────────────────────────
# For INT8 models (bitsandbytes), we must dequantize before SVD.
# For FP16 models, we just cast to float16.

import bitsandbytes as bnb
import bitsandbytes.functional as bnb_F

def get_weight_fp16(module: nn.Module) -> torch.Tensor:
    """
    Return the weight of a linear layer as a float16 CPU tensor.
    Handles bitsandbytes INT8 dequantization transparently.
    """
    w = module.weight

    # bitsandbytes >= 0.43: quant_state attribute
    if hasattr(w, "quant_state") and w.quant_state is not None:
        return bnb_F.dequantize_blockwise(w.data, w.quant_state).to(torch.float16)

    # bitsandbytes older style: CB / SCB attributes
    if hasattr(w, "CB") and w.CB is not None and hasattr(w, "SCB"):
        return (w.CB.float() * w.SCB.float().view(-1, 1) / 127.0).to(torch.float16)

    # Regular FP16 / FP32 weight
    return w.data.to(torch.float16)


def patch_weight(module: nn.Module, W_hat: torch.Tensor) -> None:
    """
    Replace a linear module's weight with W_hat (float16).
    Works for both regular and bitsandbytes INT8 layers by swapping the
    underlying module with a standard nn.Linear.
    """
    # For bitsandbytes layers, we cannot set .weight.data directly.
    # Instead we swap the parameter in-place.
    with torch.no_grad():
        if hasattr(module.weight, "quant_state") or hasattr(module.weight, "CB"):
            # Cannot patch INT8 in-place cleanly; caller should replace the module.
            raise RuntimeError(
                "INT8 layer in-place patching not supported; "
                "use replace_module() instead."
            )
        module.weight.data.copy_(W_hat.to(module.weight.device))


def replace_module_weight(parent: nn.Module, child_name: str, W_hat: torch.Tensor,
                           original_module: nn.Module) -> None:
    """
    Replace child module inside parent with a new nn.Linear whose weight is W_hat.
    Copies bias from original_module if present.
    """
    out_f, in_f = W_hat.shape
    has_bias = original_module.bias is not None
    new_mod = nn.Linear(in_f, out_f, bias=has_bias, device=W_hat.device, dtype=torch.float16)
    new_mod.weight = nn.Parameter(W_hat.to(dtype=torch.float16))
    if has_bias:
        new_mod.bias = nn.Parameter(original_module.bias.data.to(dtype=torch.float16))
    setattr(parent, child_name, new_mod)


print("Weight helpers defined.")

In [ ]:
# ── Main compression sweep ────────────────────────────────────────────────────
# For each bond dimension χ:
#   1. Compress all target layers → MPS cores
#   2. Compute reconstruction error and core param count per layer
#   3. Save cores dict to checkpoints/compressed_chi{X}/cores.pt
#   4. Verify with a test forward pass (reconstruct + patch weights)

sweep_stats = {}   # chi -> aggregated stats

for chi in BOND_DIMS_THIS_RUN:
    print(f"\n{'═'*62}")
    print(f"  Bond dimension χ = {chi}")
    print(f"{'═'*62}")

    ckpt_dir = CHECKPOINTS_DIR / f"compressed_chi{chi}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    cores_dict   = {}   # layer_name -> list of CPU core tensors
    layer_stats  = {}   # layer_name -> per-layer metrics
    total_orig   = 0
    total_cores  = 0

    t_start = time.perf_counter()

    for name, module in tqdm(targets.items(), desc=f"chi={chi}", unit="layer"):
        W_fp16 = get_weight_fp16(module)   # (out, in), FP16, on GPU
        m, n   = W_fp16.shape

        try:
            cores, sinfo = mps_decompose(W_fp16, bond_dim=chi, n_sites=N_SITES)
            W_hat        = mps_reconstruct(cores, (m, n))
            err          = frobenius_error(W_fp16, W_hat)
            n_orig_layer = m * n
            n_core_layer = count_core_params(cores)

            cores_dict[name] = [c.cpu() for c in cores]    # save on CPU
            layer_stats[name] = {
                "shape": [m, n],
                "reshape_dims": sinfo["reshape_dims"],
                "actual_ranks": sinfo["actual_ranks"],
                "n_orig": n_orig_layer,
                "n_cores": n_core_layer,
                "layer_ratio": round(n_orig_layer / max(n_core_layer, 1), 2),
                "frob_error": round(err, 6),
            }
            total_orig  += n_orig_layer
            total_cores += n_core_layer

        except Exception as exc:
            print(f"  SKIP {name}: {exc}")
            continue

    elapsed_s = time.perf_counter() - t_start

    # Aggregate error statistics
    errors = [s["frob_error"] for s in layer_stats.values()]
    ratios = [s["layer_ratio"] for s in layer_stats.values()]

    sweep_stats[chi] = {
        "bond_dim": chi,
        "n_layers_compressed": len(layer_stats),
        "total_target_params_orig": total_orig,
        "total_core_params": total_cores,
        "layer_compression_ratio_mean": float(np.mean(ratios)),
        "layer_compression_ratio_min":  float(np.min(ratios)),
        "frob_error_mean": float(np.mean(errors)),
        "frob_error_max":  float(np.max(errors)),
        "frob_error_p95":  float(np.percentile(errors, 95)),
        "elapsed_s": round(elapsed_s, 1),
    }

    print(f"\n  Layers compressed  : {len(layer_stats)}")
    print(f"  Layer param ratio  : {np.mean(ratios):.1f}× mean  "
          f"(min {np.min(ratios):.1f}×)")
    print(f"  Frob error         : {np.mean(errors):.3%} mean  "
          f"| {np.max(errors):.3%} max  "
          f"| {np.percentile(errors, 95):.3%} p95")
    print(f"  Core params        : {total_cores/1e6:.1f}M  "
          f"(vs {total_orig/1e9:.2f}B original = "
          f"{total_orig/max(total_cores,1):.1f}× compression of target layers)")
    print(f"  Wall-clock         : {elapsed_s/60:.1f} min")

    # ── §5.2 Verification: Frobenius error < 5% at χ=64 ──────────────────────
    if chi == 64:
        layers_above_5pct = [n for n, s in layer_stats.items() if s["frob_error"] > 0.05]
        if layers_above_5pct:
            print(f"  ⚠  {len(layers_above_5pct)} layers have error > 5% at χ=64:")
            for lname in layers_above_5pct[:5]:
                print(f"     {lname}: {layer_stats[lname]['frob_error']:.3%}")
        else:
            print("  [OK] All layers: Frobenius error < 5% at χ=64.")

    # ── Save cores checkpoint ─────────────────────────────────────────────────
    cores_path = ckpt_dir / "cores.pt"
    torch.save(cores_dict, cores_path)
    cores_size_mb = cores_path.stat().st_size / 1024**2

    layer_stats_path = ckpt_dir / "layer_stats.json"
    with open(layer_stats_path, "w") as f:
        json.dump(layer_stats, f, indent=2)

    print(f"  Saved: {cores_path}  ({cores_size_mb:.0f} MB)")

print("\nSweep complete.")

In [ ]:
# ── Theoretical parameter count vs. actual ────────────────────────────────────
# For an (m×n) matrix reshaped to (d0,d1,d2,d3) with bond dim χ:
#   Core params ≈ 2·d·χ  +  2·d·χ²  (boundary + interior cores)
# The compression only counts cores, not the full reconstruction W_hat.

print("Bond dim |  Core params (target layers) | Overall model ratio")
print("─────────┼──────────────────────────────┼────────────────────")

n_nontarget = n_total_params - n_target_params   # frozen part of model

for chi, st in sorted(sweep_stats.items()):
    n_cores_total     = st["total_core_params"]
    effective_total   = n_nontarget + n_cores_total
    overall_ratio     = n_total_params / max(effective_total, 1)
    print(
        f"  χ={chi:3d}   |  {n_cores_total/1e6:6.0f} M cores          "
        f"  ({st['total_target_params_orig']/1e9:.2f}B → {n_cores_total/1e6:.0f}M)"
        f"  | {overall_ratio:.2f}×"
    )

print()
print("Note: overall model ratio uses core param count for compressed layers.")
print("Phase 3 reconstructs W_hat at inference time (same shape as original).")

In [ ]:
# ── Verification: test forward pass with reconstructed weights ─────────────────
# Load cores for χ=64 (primary target), patch model weights, run one inference.
# This confirms the checkpoint is usable before we commit to downloading it.

TEST_CHI = 64 if 64 in BOND_DIMS_THIS_RUN else BOND_DIMS_THIS_RUN[-1]
test_cores_path = CHECKPOINTS_DIR / f"compressed_chi{TEST_CHI}" / "cores.pt"

print(f"Verifying χ={TEST_CHI} checkpoint...")

if not test_cores_path.exists():
    print(f"  cores.pt not found at {test_cores_path} — skipping forward-pass check.")
else:
    cores_loaded = torch.load(test_cores_path, map_location="cuda:0")

    # Patch model weights in-place (save originals, restore after test)
    originals = {}
    modules_by_name = dict(model.named_modules())

    with torch.no_grad():
        for layer_name, layer_cores in cores_loaded.items():
            mod = modules_by_name.get(layer_name)
            if mod is None:
                continue
            originals[layer_name] = mod.weight.data.clone()
            W_hat = mps_reconstruct(
                [c.to("cuda:0") for c in layer_cores],
                tuple(originals[layer_name].shape),
            ).to(mod.weight.dtype)
            mod.weight.data.copy_(W_hat)

    print(f"  Patched {len(originals)} layers.")

    # Minimal dummy forward pass (batch_size=1, seq_len=16)
    dummy_ids    = torch.randint(0, 1000, (1, 16), device="cuda:0")
    dummy_pixels = torch.zeros(1, 3, 224, 224, device="cuda:0",
                               dtype=next(model.parameters()).dtype)
    try:
        with torch.no_grad():
            out = model(
                input_ids=dummy_ids,
                pixel_values=dummy_pixels,
            )
        print(f"  [OK] Forward pass succeeded. Logits shape: {out.logits.shape}")
    except Exception as exc:
        print(f"  [FAIL] Forward pass error: {exc}")

    # Restore original weights
    with torch.no_grad():
        for layer_name, orig_w in originals.items():
            modules_by_name[layer_name].weight.data.copy_(orig_w)
    print(f"  Original weights restored.")

In [ ]:
# ── Post-compression INT8 quantization (demonstration) ───────────────────────
# After TN reconstruction, we can re-quantize W_hat to INT8 for maximum
# memory efficiency (TN compression + INT8 combined pipeline, per the plan).
# This cell demonstrates the quantization for one layer; Phase 3 applies
# it during the INT8+TN ablation evaluation.

DEMO_LAYER = next(iter(targets))   # first compression target
DEMO_CHI_Q = TEST_CHI

print(f"Post-compression INT8 quantization demo")
print(f"Layer : {DEMO_LAYER}  (χ={DEMO_CHI_Q})")

demo_cores = torch.load(
    CHECKPOINTS_DIR / f"compressed_chi{DEMO_CHI_Q}" / "cores.pt",
    map_location="cuda:0",
)[DEMO_LAYER]

mod_orig = dict(model.named_modules())[DEMO_LAYER]
W_hat = mps_reconstruct(
    [c.to("cuda:0") for c in demo_cores],
    tuple(mod_orig.weight.shape),
).to(torch.float16)

print(f"W_hat shape  : {W_hat.shape}  dtype={W_hat.dtype}")

# Quantize W_hat to INT8 using bitsandbytes blockwise scheme
W_hat_cpu  = W_hat.float().cpu().contiguous()
W_q, quant_state = bnb_F.quantize_blockwise(W_hat_cpu)

# Verify round-trip dequantization error
W_deq = bnb_F.dequantize_blockwise(W_q, quant_state).to(torch.float16).cuda()
rt_err = frobenius_error(W_hat.cpu().float(), W_deq.cpu().float())

mem_fp16   = W_hat.numel() * 2 / 1024**2   # MB
mem_int8   = W_q.numel()   * 1 / 1024**2   # MB

print(f"FP16 storage  : {mem_fp16:.1f} MB")
print(f"INT8 storage  : {mem_int8:.1f} MB  ({mem_fp16/mem_int8:.1f}× reduction)")
print(f"INT8 round-trip error: {rt_err:.4%}")
print("Combined pipeline: TN compression (structure) + INT8 (storage) confirmed.")

In [ ]:
# ── Save sweep summary ────────────────────────────────────────────────────────
sweep_output = {
    "phase": 2,
    "model": MODEL_ID,
    "n_sites": N_SITES,
    "target_suffixes": list(DEFAULT_TARGET_SUFFIXES),
    "n_target_layers": len(targets),
    "n_target_params_orig": int(n_target_params),
    "n_total_params": int(n_total_params),
    "bond_dims_processed": BOND_DIMS_THIS_RUN,
    "sweep_stats": sweep_stats,
}

# Load and merge any prior sweep stats (from a previous session)
existing_path = RESULTS_DIR / "compression_sweep_stats.json"
if existing_path.exists():
    with open(existing_path) as f:
        existing = json.load(f)
    # Merge: keep existing bond dims not processed in this run
    for chi_key, stats in existing.get("sweep_stats", {}).items():
        if int(chi_key) not in BOND_DIMS_THIS_RUN:
            sweep_output["sweep_stats"][int(chi_key)] = stats

with open(existing_path, "w") as f:
    json.dump(sweep_output, f, indent=2)

print(f"Sweep stats saved → {existing_path}")

# ── Verification checklist ────────────────────────────────────────────────────
print("\n── Verification ─────────────────────────────────────────────")

for chi in BOND_DIMS_THIS_RUN:
    st = sweep_stats.get(chi, {})
    cores_file = CHECKPOINTS_DIR / f"compressed_chi{chi}" / "cores.pt"

    ok_saved = cores_file.exists()
    ok_layers = st.get("n_layers_compressed", 0) > 100

    err_mean = st.get("frob_error_mean", 1.0)
    ok_err   = err_mean < 0.10    # mean error < 10% is healthy for all chi
    ok_err64 = (chi != 64) or (st.get("frob_error_max", 1.0) < 0.05)

    print(f"  χ={chi:3d}  "
          f"[{'OK' if ok_saved else 'FAIL'}] cores.pt saved  "
          f"[{'OK' if ok_layers else 'FAIL'}] >100 layers  "
          f"[{'OK' if ok_err else 'FAIL'}] mean err<10%  "
          f"[{'OK' if ok_err64 else 'WARN'}] max err<5% (χ=64 only)")

print("────────────────────────────────────────────────────────────")

In [ ]:
# ── Download results from Colab ────────────────────────────────────────────
# Checkpoints are already persisted on Google Drive — no download needed.
# Download only the sweep stats JSON to commit to the repo.
from google.colab import files

files.download(str(RESULTS_DIR / "compression_sweep_stats.json"))

print("\nCheckpoints are saved on Google Drive at:")
print(f"  {CHECKPOINTS_BASE}")
print("\nCommit the results JSON:")
print("  git add results/compression_sweep_stats.json")
print("  git commit -m '[phase2] add TN compression sweep stats'")
print("\nPhase 3 will mount the same Drive path to read checkpoints directly.")

## Results Summary

After running all cells, the following artefacts are produced:

| Artefact | Contents |
|---|---|
| `checkpoints/compressed_chi{X}/cores.pt` | `{layer_name: [core_tensor, ...]}` for all compressed layers |
| `checkpoints/compressed_chi{X}/layer_stats.json` | Per-layer compression ratio and Frobenius error |
| `results/compression_sweep_stats.json` | Aggregated stats across all χ values |

### How Phase 3 uses the checkpoints

```python
from vlam_compress.compress import mps_reconstruct

cores_dict = torch.load("checkpoints/compressed_chi64/cores.pt")
for layer_name, cores in cores_dict.items():
    W_hat = mps_reconstruct(cores, original_shape)
    # Patch model layer with W_hat for inference
```

### Challenge sections satisfied

- **§5.2** Functional QI component: MPS/MPO decomposition implemented and validated
- **§5.2** quimb used for TN structure bookkeeping (cell 9)
- **§4.1** Reduced parameter count: core params << original matrix params
- **§5.1** Model footprint reduction demonstrated in the parameter count table
- **§5.5** Quantum Justification: TN decomposition is the QI component ablated in Phase 3
- **§5.3** Reconstruction error reported per layer (Frobenius norm ratio)